# Chapter 5 — Q-Learning from Scratch## Concept Notebook[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**GPU optional. Runtime: ~20 minutes CPU / ~10 minutes GPU.**This notebook builds Q-Learning and DQN step by step, from the tabular update rule to a neural network Q-function on CartPole.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('Ready. If torch is available, DQN section will use it.')
try:
    import torch
    import torch.nn as nn
    print(f'PyTorch {torch.__version__} available. GPU:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not found. Only tabular Q-Learning sections will run.')

## 5.1 The Q-Learning Update RuleFrom Chapter 5: the TD update for Q(s,a):```Q(s,a) ← Q(s,a) + α [r + γ max_{a'} Q(s',a') - Q(s,a)]```This is the Bellman optimality equation applied as an update rule. Let's implement it on GridWorld.

In [ ]:
from collections import defaultdict
import random

size = 5
goal = (4, 4)
actions = ['UP','DOWN','LEFT','RIGHT']
action_deltas = {'UP':(-1,0),'DOWN':(1,0),'LEFT':(0,-1),'RIGHT':(0,1)}

def step(pos, action):
    dr, dc = action_deltas[action]
    nr, nc = pos[0]+dr, pos[1]+dc
    if 0 <= nr < size and 0 <= nc < size:
        npos = (nr, nc)
        reward = 10 if npos == goal else -1
    else:
        npos = pos
        reward = -5
    return npos, reward, npos == goal

# Q-Learning
Q = defaultdict(lambda: defaultdict(float))
alpha, gamma, epsilon = 0.2, 0.9, 0.4
rewards_history = []

for episode in range(3000):
    pos = (0, 0)
    ep_reward = 0
    for _ in range(50):
        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            action = max(actions, key=lambda a: Q[pos][a])
        
        next_pos, reward, done = step(pos, action)
        
        # Q-Learning update (off-policy TD)
        best_next = max(Q[next_pos][a] for a in actions)
        Q[pos][action] += alpha * (reward + gamma * best_next - Q[pos][action])
        
        ep_reward += reward
        pos = next_pos
        if done: break
    
    rewards_history.append(ep_reward)
    epsilon = max(0.05, epsilon * 0.999)

window = 100
smoothed = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(smoothed, color='steelblue', linewidth=1.5)
ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Reward')
ax.set_title('Q-Learning on GridWorld: Learning Curve')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.2 DQN: Replacing the Q-Table with a Neural NetworkWhen the state space is continuous or very large, we cannot store a table. DQN approximates Q(s,a) with a neural network.Key additions vs tabular Q-Learning:1. **Experience replay**: store transitions in a replay buffer, sample mini-batches2. **Target network**: use a frozen copy of the network for stable TD targets

In [ ]:
try:
    import torch
    import torch.nn as nn
    import gymnasium as gym
    from collections import deque
    
    class DQN(nn.Module):
        def __init__(self, obs_dim, n_actions):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(obs_dim, 128), nn.ReLU(),
                nn.Linear(128, 128), nn.ReLU(),
                nn.Linear(128, n_actions)
            )
        def forward(self, x): return self.net(x)
    
    env = gym.make('CartPole-v1')
    obs_dim = env.observation_space.shape[0]  # 4
    n_actions = env.action_space.n  # 2
    
    policy_net = DQN(obs_dim, n_actions)
    target_net = DQN(obs_dim, n_actions)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    
    replay_buffer = deque(maxlen=10000)
    optimizer = torch.optim.Adam(policy_net.parameters(), lr=1e-3)
    
    print(f'DQN ready. Obs: {obs_dim}D, Actions: {n_actions}')
    print('Architecture:', policy_net)
except ImportError as e:
    print(f'Skipping DQN section: {e}')

In [ ]:
try:
    # DQN Training Loop
    BATCH_SIZE, GAMMA, EPSILON, TARGET_UPDATE = 64, 0.99, 0.9, 10
    rewards_dqn = []
    
    for episode in range(200):
        obs, _ = env.reset()
        ep_reward = 0
        while True:
            # Epsilon-greedy
            if random.random() < EPSILON:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    q_vals = policy_net(torch.FloatTensor(obs))
                    action = q_vals.argmax().item()
            
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            replay_buffer.append((obs, action, reward, next_obs, done))
            obs = next_obs
            ep_reward += reward
            
            if len(replay_buffer) >= BATCH_SIZE:
                batch = random.sample(replay_buffer, BATCH_SIZE)
                obs_b, act_b, rew_b, nobs_b, done_b = zip(*batch)
                obs_t = torch.FloatTensor(obs_b)
                act_t = torch.LongTensor(act_b)
                rew_t = torch.FloatTensor(rew_b)
                nobs_t = torch.FloatTensor(nobs_b)
                done_t = torch.FloatTensor(done_b)
                
                current_q = policy_net(obs_t).gather(1, act_t.unsqueeze(1)).squeeze()
                with torch.no_grad():
                    next_q = target_net(nobs_t).max(1)[0]
                target_q = rew_t + GAMMA * next_q * (1 - done_t)
                
                loss = nn.functional.mse_loss(current_q, target_q)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            if done: break
        
        rewards_dqn.append(ep_reward)
        EPSILON = max(0.05, EPSILON * 0.99)
        if episode % TARGET_UPDATE == 0:
            target_net.load_state_dict(policy_net.state_dict())
    
    print(f'Final 20-episode mean: {np.mean(rewards_dqn[-20:]):.1f}')
except Exception as e:
    print(f'DQN training skipped: {e}')

## ✅ Key Takeaways1. Q-Learning: off-policy TD — learns Q* regardless of the behaviour policy2. DQN: Q-table → neural network + experience replay + target network3. Experience replay breaks temporal correlations between consecutive updates4. Target network provides stable training targets (updated every K steps, not every step)